In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("data/Indian_ODI_Cricketers_With_Role.csv")

print("Original shape:", df.shape)
df.head()

Original shape: (250, 20)


,No,Name,First,Last,Mat,Inn,NO.1,Runs,HS,Avg,Balls,Mdn,Runs_1,Wkt,BBM,Avg_2,Ca,St,Selected,Role
0,1,Syed Abid Ali,1974,1975,5,3,0,93,70,31.00,336,10,187,7,Feb-22,26.71,0,0,0,Batsman
1,2,Bishan Singh Bedi,1974,1979,10,7,2,31,13,6.20,590,17,340,7,Feb-44,48.57,4,0,0,Batsman
2,3,Farokh Engineer,1974,1975,5,4,1,114,54*,38.00,0,0,0,0,NaN,0.00,3,1,0,Wicketkeeper
3,4,Sunil Gavaskar,1974,1987,108,102,14,3092,103*,35.13,20,0,25,1,01-Oct,25.00,22,0,0,Batsman
4,5,Madan Lal,1974,1987,67,35,14,401,53*,19.09,3164,44,2137,73,Apr-20,29.27,18,0,0,Bowler


In [3]:
df = df[df["Last"] >= 2020].copy()

print("After filtering Last >= 2020:", df.shape)
df[["Name", "Last", "Role"]].head()

After filtering Last >= 2020: (45, 20)


,Name,Last,Role
167,Rohit Sharma,2023,Batsman
174,Virat Kohli,2023,Batsman
176,Ravindra Jadeja,2023,AllRounder
184,Ravichandran Ashwin,2022,Bowler
187,Shikhar Dhawan,2022,Batsman


In [4]:
numeric_cols = [
    'Mat',
    'Inn',
    'NO.1',
    'Runs',
    'Avg',
    'Balls',
    'Mdn',
    'Runs_1',
    'Wkt',
    'Avg_2',
    'Ca',
    'St',
    'Selected'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols] = df[numeric_cols].fillna(0)

print(df[numeric_cols].head())

     Mat  Inn  NO.1   Runs    Avg  Balls  Mdn  Runs_1  Wkt   Avg_2   Ca  St  \
167  243  236    34   9825  48.63    593    2     515    8   64.37   87   0   
174  274  265    40  12898  57.32    641    1     665    4  166.25  141   0   
176  174  118    41   2526  32.80   8725   50    7142  191   37.39   65   0   
184  113   63    20    707  16.07   6141   36    5058  151   33.49   30   0   
187  167  164    10   6793  44.11      0    0       0    0    0.00   83   0   

     Selected  
167         1  
174         1  
176         1  
184         1  
187         1  


In [5]:
# Avoid division by zero
df['Balls'] = df['Balls'].replace(0, 1)
df['Inn'] = df['Inn'].replace(0, 1)

# Feature engineering
df['strike_rate'] = (df['Runs'] / df['Balls']) * 100
df['consistency'] = df['Runs'] / df['Inn']
df['economy'] = (df['Runs_1'] / df['Balls']) * 6

print(df[['Name', 'strike_rate', 'consistency', 'economy']].head())

                    Name    strike_rate  consistency   economy
167        Rohit Sharma     1656.829680    41.631356  5.210793
174         Virat Kohli     2012.168487    48.671698  6.224649
176      Ravindra Jadeja      28.951289    21.406780  4.911404
184  Ravichandran Ashwin      11.512783    11.222222  4.941866
187      Shikhar Dhawan   679300.000000    41.420732  0.000000


In [6]:
features = [
    'Avg',
    'Runs',
    'strike_rate',
    'consistency',
    'Wkt',
    'Avg_2',
    'economy',
    'Mat',
    'Mdn'
]

In [14]:
def generate_role_predictions(role_name):
    role_df = df[
        (df['Role'] == role_name) &
        (df['Last'] >= 2020)
    ].copy()

    # -----------------------------
    # Minimum match filters by role
    # -----------------------------
    if role_name == "Batsman":
        role_df = role_df[role_df['Mat'] >= 10]

    elif role_name == "Bowler":
        role_df = role_df[role_df['Mat'] >= 10]

    elif role_name == "AllRounder":
        role_df = role_df[role_df['Mat'] >= 10]

    elif role_name == "Wicketkeeper":
        role_df = role_df[role_df['Mat'] >= 5]

    print(f"\nPlayers in {role_name} after filters:", len(role_df))

    if len(role_df) == 0:
        return pd.DataFrame()

    # Feature engineering
    role_df['Balls'] = role_df['Balls'].replace(0, 1)
    role_df['Inn'] = role_df['Inn'].replace(0, 1)

    role_df['strike_rate'] = (role_df['Runs'] / role_df['Balls']) * 100
    role_df['consistency'] = role_df['Runs'] / role_df['Inn']
    role_df['economy'] = (role_df['Runs_1'] / role_df['Balls']) * 6

    features = [
        'Mat', 'Inn', 'Runs', 'Avg',
        'Wkt', 'Avg_2',
        'strike_rate', 'consistency', 'economy'
    ]

    X = role_df[features].fillna(0)
    y = role_df['Selected']

    # If only one class present, assign fallback probability
    if y.nunique() < 2:
        role_df['selection_probability'] = y.astype(float)
        return role_df.sort_values(
            by='selection_probability',
            ascending=False
        )

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
    model.fit(X_scaled, y)

    role_df['selection_probability'] = model.predict_proba(X_scaled)[:, 1]

    role_df = role_df.sort_values(
        by='selection_probability',
        ascending=False
    )

    return role_df

In [15]:
best_batsmen = generate_role_predictions("Batsman")

best_batsmen[
    ['Name', 'Role', 'Last', 'selection_probability']
].head(5)


Players in Batsman after filters: 11


,Name,Role,Last,selection_probability
167,Rohit Sharma,Batsman,2023,0.995
174,Virat Kohli,Batsman,2023,0.995
187,Shikhar Dhawan,Batsman,2022,0.970
218,Shreyas Iyer,Batsman,2023,0.925
204,Kedar Jadhav,Batsman,2020,0.865


In [16]:
best_bowlers = generate_role_predictions("Bowler")

best_bowlers[
    ['Name', 'Role', 'Last', 'selection_probability']
].head(5)


Players in Bowler after filters: 8


,Name,Role,Last,selection_probability
184,Ravichandran Ashwin,Bowler,2022,0.950
193,Bhuvneshwar Kumar,Bowler,2022,0.885
194,Mohammed Shami,Bowler,2023,0.775
209,Jasprit Bumrah,Bowler,2022,0.685
216,Kuldeep Yadav,Bowler,2023,0.200


In [17]:
best_allrounders = generate_role_predictions("AllRounder")

best_allrounders[
    ['Name', 'Role', 'Last', 'selection_probability']
].head()


Players in AllRounder after filters: 4


,Name,Role,Last,selection_probability
176,Ravindra Jadeja,AllRounder,2023,0.90
214,Hardik Pandya,AllRounder,2023,0.81
201,Axar Patel,AllRounder,2023,0.18
219,Washington Sundar,AllRounder,2023,0.18


In [18]:
best_wicketkeepers = generate_role_predictions("Wicketkeeper")
best_wicketkeepers[['Name', 'Role', 'Last', 'Selected', 'selection_probability']].head(10)


Players in Wicketkeeper after filters: 4


,Name,Role,Last,Selected,selection_probability
212,KL Rahul,Wicketkeeper,2023,1,0.730
240,Sanju Samson,Wicketkeeper,2022,1,0.700
234,Ishan Kishan,Wicketkeeper,2023,0,0.225
223,Rishabh Pant,Wicketkeeper,2022,0,0.095


In [19]:
final_squad = pd.concat([
    best_batsmen.head(5),
    best_bowlers.head(5),
    best_allrounders.head(3),
    best_wicketkeepers.head(2)
])

final_squad = final_squad.sort_values(
    by='selection_probability',
    ascending=False
)

final_squad[['Name', 'Role', 'Last', 'Selected', 'selection_probability']]

,Name,Role,Last,Selected,selection_probability
167,Rohit Sharma,Batsman,2023,1,0.995
174,Virat Kohli,Batsman,2023,1,0.995
187,Shikhar Dhawan,Batsman,2022,1,0.970
184,Ravichandran Ashwin,Bowler,2022,1,0.950
218,Shreyas Iyer,Batsman,2023,1,0.925
176,Ravindra Jadeja,AllRounder,2023,1,0.900
193,Bhuvneshwar Kumar,Bowler,2022,1,0.885
204,Kedar Jadhav,Batsman,2020,1,0.865
214,Hardik Pandya,AllRounder,2023,1,0.810
194,Mohammed Shami,Bowler,2023,1,0.775


In [21]:
# ============================================
# FINAL SQUAD RULE:
# 4 batsmen + 2 allrounders + 1 wicketkeeper + 4 bowlers
# reserves: 1 batsman + 1 wicketkeeper + 1 allrounder + 1 bowler
# ============================================

# -------------------------
# PLAYING XI
# -------------------------

main_batsmen = best_batsmen.head(4).copy()
main_allrounders = best_allrounders.head(2).copy()
main_wicketkeeper = best_wicketkeepers.head(1).copy()
main_bowlers = best_bowlers.head(4).copy()

playing_xi = pd.concat([
    main_batsmen,        # positions 1,2,3,4
    main_allrounders,    # positions 5,6
    main_wicketkeeper,   # position 7
    main_bowlers         # positions 8,9,10,11
], ignore_index=True)

# Remove duplicates if same player appears more than once
playing_xi = playing_xi.drop_duplicates(subset='Name', keep='first')

# -------------------------
# IF XI < 11 AFTER DUPLICATE REMOVAL, FILL IT
# -------------------------

needed_xi = 11 - len(playing_xi)

if needed_xi > 0:
    remaining_players = pd.concat([
        best_batsmen,
        best_allrounders,
        best_wicketkeepers,
        best_bowlers
    ], ignore_index=True)

    remaining_players = remaining_players.drop_duplicates(subset='Name')
    remaining_players = remaining_players[
        ~remaining_players['Name'].isin(playing_xi['Name'])
    ]

    xi_fill = remaining_players.head(needed_xi).copy()
    playing_xi = pd.concat([playing_xi, xi_fill], ignore_index=True)

# -------------------------
# RESERVES
# -------------------------

reserve_batsman = best_batsmen.iloc[4:5].copy()             # 5th batsman
reserve_wicketkeeper = best_wicketkeepers.iloc[1:2].copy()  # 2nd wicketkeeper
reserve_allrounder = best_allrounders.iloc[2:3].copy()      # 3rd allrounder
reserve_bowler = best_bowlers.iloc[4:5].copy()              # 5th bowler

reserves = pd.concat([
    reserve_batsman,
    reserve_wicketkeeper,
    reserve_allrounder,
    reserve_bowler
], ignore_index=True)

# Remove anyone already in playing XI
reserves = reserves[
    ~reserves['Name'].isin(playing_xi['Name'])
].copy()

# -------------------------
# IF RESERVES < 4, FILL THEM
# -------------------------

needed_reserves = 4 - len(reserves)

if needed_reserves > 0:
    remaining_players = pd.concat([
        best_batsmen,
        best_allrounders,
        best_wicketkeepers,
        best_bowlers
    ], ignore_index=True)

    remaining_players = remaining_players.drop_duplicates(subset='Name')
    remaining_players = remaining_players[
        ~remaining_players['Name'].isin(playing_xi['Name'])
    ]
    remaining_players = remaining_players[
        ~remaining_players['Name'].isin(reserves['Name'])
    ]

    reserve_fill = remaining_players.head(needed_reserves).copy()
    reserves = pd.concat([reserves, reserve_fill], ignore_index=True)

# -------------------------
# FINAL 15-MAN SQUAD
# -------------------------

final_squad = pd.concat([
    playing_xi,
    reserves
], ignore_index=True)

# Keep only required columns
final_squad = final_squad[
    ['Name', 'Role', 'selection_probability']
]

final_squad

,Name,Role,selection_probability
0,Rohit Sharma,Batsman,0.995
1,Virat Kohli,Batsman,0.995
2,Shikhar Dhawan,Batsman,0.970
3,Shreyas Iyer,Batsman,0.925
4,Ravindra Jadeja,AllRounder,0.900
5,Hardik Pandya,AllRounder,0.810
6,KL Rahul,Wicketkeeper,0.730
7,Ravichandran Ashwin,Bowler,0.950
8,Bhuvneshwar Kumar,Bowler,0.885
9,Mohammed Shami,Bowler,0.775


In [62]:
# =========================
# BUILD FINAL SQUAD IN MATCH ORDER
# =========================

# Main playing XI
playing_batsmen = best_batsmen.head(4).copy()         # top-order batsmen
playing_wk = best_wicketkeepers.head(1).copy()        # 1 wicketkeeper in XI
playing_allrounders = best_allrounders.head(2).copy() # middle-order all-rounders
playing_bowlers = best_bowlers.head(4).copy()         # main bowlers

# Combine main XI in cricket-style order
playing_xi = pd.concat([
    playing_batsmen,
    playing_wk,
    playing_allrounders,
    playing_bowlers
], ignore_index=True)

# Remove duplicates if any player appears in multiple groups
playing_xi = playing_xi.drop_duplicates(subset='Name', keep='first')

# If XI becomes less than 11 due to duplicates, fill from remaining best players
needed_xi = 11 - len(playing_xi)

if needed_xi > 0:
    remaining_for_xi = pd.concat([
        best_batsmen,
        best_wicketkeepers,
        best_allrounders,
        best_bowlers
    ], ignore_index=True)

    remaining_for_xi = remaining_for_xi.drop_duplicates(subset='Name')
    remaining_for_xi = remaining_for_xi[
        ~remaining_for_xi['Name'].isin(playing_xi['Name'])
    ]

    xi_fill = remaining_for_xi.head(needed_xi)
    playing_xi = pd.concat([playing_xi, xi_fill], ignore_index=True)

# =========================
# RESERVES
# =========================

remaining_players = pd.concat([
    best_batsmen,
    best_wicketkeepers,
    best_allrounders,
    best_bowlers
], ignore_index=True)

remaining_players = remaining_players.drop_duplicates(subset='Name')
remaining_players = remaining_players[
    ~remaining_players['Name'].isin(playing_xi['Name'])
]

reserves = remaining_players.head(4).copy()

# =========================
# FINAL 15-MAN SQUAD
# =========================

final_squad = pd.concat([
    playing_xi,
    reserves
], ignore_index=True)

# Display only the required columns
final_squad[['Name', 'Role', 'selection_probability']]

,Name,Role,selection_probability
0,Rohit Sharma,Batsman,0.985
1,Virat Kohli,Batsman,0.985
2,Shikhar Dhawan,Batsman,0.985
3,Shreyas Iyer,Batsman,0.915
4,KL Rahul,Wicketkeeper,0.730
5,Hardik Pandya,AllRounder,0.875
6,Krunal Pandya,AllRounder,0.875
7,Ravichandran Ashwin,Bowler,0.930
8,Bhuvneshwar Kumar,Bowler,0.900
9,Mohammed Shami,Bowler,0.795


In [64]:
# =========================
# MAIN PLAYING CORE / SQUAD ORDER
# =========================

# First choose role-wise players
main_batsmen = best_batsmen.head(5).copy()
main_wicketkeepers = best_wicketkeepers.head(2).copy()
main_allrounders = best_allrounders.head(3).copy()
main_bowlers = best_bowlers.head(5).copy()

# Add display category so we know the section
main_batsmen["Squad_Group"] = "Batsmen / Top Order"
main_wicketkeepers["Squad_Group"] = "Wicketkeepers"
main_allrounders["Squad_Group"] = "AllRounders / Middle Order"
main_bowlers["Squad_Group"] = "Bowlers"

# Combine in desired order
final_squad = pd.concat([
    main_batsmen,
    main_wicketkeepers,
    main_allrounders,
    main_bowlers
], ignore_index=True)

# Remove duplicates if any player appears in more than one role block
final_squad = final_squad.drop_duplicates(subset="Name", keep="first")

# If due to duplicates squad becomes less than 15,
# fill remaining places from leftover best players as reserves
needed = 15 - len(final_squad)

if needed > 0:
    remaining_players = pd.concat([
        best_batsmen,
        best_wicketkeepers,
        best_allrounders,
        best_bowlers
    ], ignore_index=True)

    remaining_players = remaining_players.drop_duplicates(subset="Name")
    remaining_players = remaining_players[
        ~remaining_players["Name"].isin(final_squad["Name"])
    ].copy()

    reserves = remaining_players.head(needed).copy()
    reserves["Squad_Group"] = "Reserves"

    final_squad = pd.concat([final_squad, reserves], ignore_index=True)

# Display final squad in conventional order
final_squad[
    ['Name', 'Role', 'Last', 'Selected', 'selection_probability', 'Squad_Group']

SyntaxError: incomplete input (1701660365.py, line 52)

In [22]:
# ============================================
# FINAL SQUAD RULE:
# 4 batsmen + 2 allrounders + 1 wicketkeeper + 4 bowlers
# reserves: 1 batsman + 1 wicketkeeper + 1 allrounder + 1 bowler
# ============================================

# -------------------------
# PLAYING XI
# -------------------------

main_batsmen = best_batsmen.head(4).copy()
main_allrounders = best_allrounders.head(2).copy()
main_wicketkeeper = best_wicketkeepers.head(1).copy()
main_bowlers = best_bowlers.head(4).copy()

playing_xi = pd.concat([
    main_batsmen,        # positions 1,2,3,4
    main_allrounders,    # positions 5,6
    main_wicketkeeper,   # position 7
    main_bowlers         # positions 8,9,10,11
], ignore_index=True)

# Remove duplicates if same player appears more than once
playing_xi = playing_xi.drop_duplicates(subset='Name', keep='first')

# -------------------------
# IF XI < 11 AFTER DUPLICATE REMOVAL, FILL IT
# -------------------------

needed_xi = 11 - len(playing_xi)

if needed_xi > 0:
    remaining_players = pd.concat([
        best_batsmen,
        best_allrounders,
        best_wicketkeepers,
        best_bowlers
    ], ignore_index=True)

    remaining_players = remaining_players.drop_duplicates(subset='Name')
    remaining_players = remaining_players[
        ~remaining_players['Name'].isin(playing_xi['Name'])
    ]

    xi_fill = remaining_players.head(needed_xi).copy()
    playing_xi = pd.concat([playing_xi, xi_fill], ignore_index=True)

# -------------------------
# RESERVES
# -------------------------

reserve_batsman = best_batsmen.iloc[4:5].copy()             # 5th batsman
reserve_wicketkeeper = best_wicketkeepers.iloc[1:2].copy()  # 2nd wicketkeeper
reserve_allrounder = best_allrounders.iloc[2:3].copy()      # 3rd allrounder
reserve_bowler = best_bowlers.iloc[4:5].copy()              # 5th bowler

reserves = pd.concat([
    reserve_batsman,
    reserve_wicketkeeper,
    reserve_allrounder,
    reserve_bowler
], ignore_index=True)

# Remove anyone already in playing XI
reserves = reserves[
    ~reserves['Name'].isin(playing_xi['Name'])
].copy()

# -------------------------
# IF RESERVES < 4, FILL THEM
# -------------------------

needed_reserves = 4 - len(reserves)

if needed_reserves > 0:
    remaining_players = pd.concat([
        best_batsmen,
        best_allrounders,
        best_wicketkeepers,
        best_bowlers
    ], ignore_index=True)

    remaining_players = remaining_players.drop_duplicates(subset='Name')
    remaining_players = remaining_players[
        ~remaining_players['Name'].isin(playing_xi['Name'])
    ]
    remaining_players = remaining_players[
        ~remaining_players['Name'].isin(reserves['Name'])
    ]

    reserve_fill = remaining_players.head(needed_reserves).copy()
    reserves = pd.concat([reserves, reserve_fill], ignore_index=True)

# -------------------------
# FINAL 15-MAN SQUAD
# -------------------------

final_squad = pd.concat([
    playing_xi,
    reserves
], ignore_index=True)

# Keep only required columns
final_squad = final_squad[
    ['Name', 'Role', 'selection_probability']
]

final_squad

,Name,Role,selection_probability
0,Rohit Sharma,Batsman,0.995
1,Virat Kohli,Batsman,0.995
2,Shikhar Dhawan,Batsman,0.970
3,Shreyas Iyer,Batsman,0.925
4,Ravindra Jadeja,AllRounder,0.900
5,Hardik Pandya,AllRounder,0.810
6,KL Rahul,Wicketkeeper,0.730
7,Ravichandran Ashwin,Bowler,0.950
8,Bhuvneshwar Kumar,Bowler,0.885
9,Mohammed Shami,Bowler,0.775
